In [ ]:
import pandas as pd
import json
import html
from bs4 import BeautifulSoup

# 1. Đọc lại file gốc (file có 4 dòng/câu)
df_full = pd.read_csv(r'D:\IntelligentTesting\intelligent-testing\dataset\Question.csv')

# 2. Gom nhóm để đưa content vào mảng
# Chúng ta sẽ gom theo question_id và tạo danh sách nội dung tương ứng
df_questions_fixed = df_full.groupby('question_id').agg(
    question_content=('question_content', 'first'), # Lấy nội dung câu hỏi
    all_option_ids=('option_id', list),             # Danh sách ID đáp án
    all_options_content=('option_content', list),   # MỚI: Danh sách NỘI DUNG đáp án
    correct_option_ids=('option_correct', lambda x: [id_val for id_val, corr in zip(df_full.loc[x.index, 'option_id'], x) if corr == 1]),
    skill_ids=('skill_ids', lambda x: list(set(x)))  # Gom kỹ năng (loại bỏ trùng lặp)
).reset_index()

# 3. Xử lý trường hợp thiếu kỹ năng (Gán mặc định 8)
df_questions_fixed['skill_ids'] = df_questions_fixed['skill_ids'].apply(lambda x: x if len(x) > 0 else [8])

# 3.5. CHUẨN HÓA KỸ NĂNG:
#   - Tách mã gộp: 46 -> [4, 6], 64 -> [6, 4] (câu cần CẢ HAI kỹ năng 4 và 6).
#   - Câu thiếu kỹ năng (NaN/None) -> gán 8 ('null'), đồng bộ với bảng skills và skill_seq của sessions.
#   - Lưu ý: 15 là kỹ năng hợp lệ (Chương 8: Pointer) nên KHÔNG bị tách.
VALID_SKILL_IDS = {1, 2, 3, 4, 5, 6, 7, 8, 15}

def expand_skill_ids(skill_list):
    if not isinstance(skill_list, list):
        return [8]
    result = []
    for s in skill_list:
        # Thiếu kỹ năng (NaN/None) -> gán kỹ năng 'null' (8)
        if s is None or (isinstance(s, float) and pd.isna(s)):
            result.append(8)
            continue
        v = int(s)
        if v in VALID_SKILL_IDS:
            result.append(v)
        else:
            # Mã gộp (vd 46, 64): tách thành từng chữ số kỹ năng hợp lệ
            digits = [int(ch) for ch in str(v)]
            if all(d in VALID_SKILL_IDS for d in digits):
                result.extend(digits)
            else:
                result.append(v)  # Không nhận dạng được -> giữ nguyên để rà soát
    # Loại trùng nhưng giữ nguyên thứ tự xuất hiện
    seen, deduped = set(), []
    for v in result:
        if v in seen:
            continue
        seen.add(v)
        deduped.append(v)
    return deduped

df_questions_fixed['skill_ids'] = df_questions_fixed['skill_ids'].apply(expand_skill_ids)

def clean_html(text):
    if not isinstance(text, str): return ""
    # 1. Giải mã các thực thể HTML (ví dụ: &igrave; -> ì)
    text = html.unescape(text)
    # 2. Dùng BeautifulSoup để tách lấy text thuần từ các thẻ
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ", strip=True)

# Áp dụng hàm clean cho cả nội dung câu hỏi và các đáp án

# 1. Dọn dẹp question_content
df_questions_fixed['question_content'] = df_questions_fixed['question_content'].apply(clean_html)

# 2. Dọn dẹp all_options_content (lấy text trong thẻ <p>)
df_questions_fixed['all_options_content'] = df_questions_fixed['all_options_content'].apply(
    lambda options: [clean_html(opt) for opt in options]
)

# 4. Xuất file JSON (ghi đè đúng file dùng để nạp vào DB)
df_questions_fixed.to_json('questions_db_ready.json', orient='records', force_ascii=False, indent=4)

print("✅ Đã cập nhật xong ngân hàng đề (đã tách kỹ năng gộp 46/64, gán 8 cho câu thiếu kỹ năng)!")
print("🔍 XEM CẤU TRÚC 1 CÂU HỎI MỚI:")
print(df_questions_fixed.head(1).to_dict(orient='records')[0])


In [ ]:
import pandas as pd

print("🔍 ĐANG TRUY VẾT CÁC CÂU HỎI THIẾU KỸ NĂNG (ID = 8)...")

# 1. Đọc lại file ngân hàng đề đã chuẩn bị ở bước trước
df_questions = pd.read_json('questions_db_ready.json')

# 2. Lọc ra những dòng mà mảng skill_ids có chứa số 8 (kỹ năng 'null')
df_cau_hoi_loi = df_questions[df_questions['skill_ids'].apply(lambda x: 8 in x)]

# 3. Hiển thị thông tin cơ bản lên màn hình
so_luong = len(df_cau_hoi_loi)
print(f"⚠️ Phát hiện {so_luong} câu hỏi đang bị gán kỹ năng Null [8]:\n")

# Chỉ chọn in ra ID và kỹ năng cho gọn
bang_tom_tat = df_cau_hoi_loi[['question_id', 'skill_ids']]
print(bang_tom_tat.to_string(index=False))

# 4. Xuất riêng rổ dữ liệu này ra một file CSV để bạn tiện rà soát thủ công
output_file = 'danh_sach_cau_hoi_thieu_skill.csv'

# Dùng utf-8-sig để Excel mở lên không bị lỗi font tiếng Việt
df_cau_hoi_loi.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n💾 Đã trích xuất danh sách này ra file: {output_file}")


In [ ]:
import pandas as pd

print("🚀 ĐANG XỬ LÝ BẢNG SKILLS TỪ FILE CSV...")

# 1. Đọc file CSV danh mục kỹ năng của bạn
df_skills = pd.read_csv(r'D:\IntelligentTesting\intelligent-testing\dataset\Skill.csv')

# Đổi tên cột 'id_skill' (như trong ảnh của bạn) thành 'skill_id' cho chuẩn Schema DB
if 'id_skill' in df_skills.columns:
    df_skills = df_skills.rename(columns={'id_skill': 'skill_id'})

# 2. Chuẩn bị bản ghi số 8 (Null/Unknown)
new_skill = pd.DataFrame([{
    'skill_id': 8,
    'skill_name': 'null'
}])

# 3. Gắn thêm (Concat) bản ghi mới vào bảng hiện tại
if not (df_skills['skill_id'] == 8).any():
    df_skills = pd.concat([df_skills, new_skill], ignore_index=True)

# Sắp xếp lại theo ID từ nhỏ đến lớn cho gọn gàng
df_skills = df_skills.sort_values(by='skill_id').reset_index(drop=True)

# 4. Xuất ra file JSON chuẩn bị cho Database
output_file = 'skills_db_ready.json'
df_skills.to_json(output_file, orient='records', force_ascii=False, indent=4)

print(f"✅ Hoàn tất! Đã gắn thêm thành công. Tổng số kỹ năng hiện tại: {len(df_skills)}")
print(f"💾 Dữ liệu đã được lưu thành công vào file: {output_file}\n")

print("🔍 XEM TRƯỚC BẢNG DỮ LIỆU ĐÃ ĐƯỢC CẬP NHẬT:")
print(df_skills.to_string(index=False))

🚀 ĐANG XỬ LÝ BẢNG SKILLS TỪ FILE CSV...
✅ Hoàn tất! Đã gắn thêm thành công. Tổng số kỹ năng hiện tại: 9
💾 Dữ liệu đã được lưu thành công vào file: skills_db_ready.json

🔍 XEM TRƯỚC BẢNG DỮ LIỆU ĐÃ ĐƯỢC CẬP NHẬT:
 skill_id                           skill_name
        1                 Chương 1: Extern C++
        2                        Chương 2: OOP
        3       Chương 3: Operator Overloading
        4 Chương 4: Inheritance & Polymorphism
        5               Chương 5: Polymorphism
        6                   Chương 5: Template
        7                  Chương 6: Exception
        8                                 null
       15                    Chương 8: Pointer


In [ ]:
import pandas as pd
import json

print("🚀 BẮT ĐẦU XÂY DỰNG TẬP DỮ LIỆU HUẤN LUYỆN (SEQUENCES)...")

# ==========================================
# 1. ĐỌC DỮ LIỆU (Cập nhật đường dẫn của bạn)
# ==========================================
df_history = pd.read_csv(r'D:\IntelligentTesting\intelligent-testing\dataset\Session.csv')

# Đọc file Json bảng câu hỏi mà chúng ta đã gọt dũa ở bước trước
# (Đảm bảo file questions_db_ready.json nằm cùng thư mục, hoặc thay bằng đường dẫn tuyệt đối)
df_questions = pd.read_json('questions_db_ready.json')


# ==========================================
# 2. HỢP NHẤT VÀ XỬ LÝ SƠ BỘ
# ==========================================
df_history['options'] = df_history['options'].astype(str)

# Nối 2 bảng dựa trên question_id (Chỉ lấy cột đáp án đúng và kỹ năng mang sang)
df_merged = pd.merge(
    df_history, 
    df_questions[['question_id', 'correct_option_ids', 'skill_ids']], 
    on='question_id', 
    how='left'
)

# Chống lỗi NaN: Nếu câu hỏi nào trong lịch sử không tồn tại trong ngân hàng đề
df_merged['correct_option_ids'] = df_merged['correct_option_ids'].apply(lambda x: x if isinstance(x, list) else [])
df_merged['skill_ids'] = df_merged['skill_ids'].apply(lambda x: x if isinstance(x, list) else [8])


# ==========================================
# 3. THUẬT TOÁN TỰ ĐỘNG CHẤM ĐIỂM (Auto-Grading)
# ==========================================
def cham_diem(row):
    try:
        # Tách chuỗi user chọn (VD: "174, 175") thành mảng số nguyên [174, 175]
        selected = [int(x.strip()) for x in str(row['options']).split(',') if x.strip()]
    except:
        selected = []
    
    correct = row['correct_option_ids']
    
    # So sánh 2 mảng (Dùng set để không phân biệt thứ tự trước sau)
    # Nếu user chọn đủ và đúng các đáp án -> Trả về 1 (Đúng), ngược lại 0 (Sai)
    if set(selected) == set(correct) and len(selected) > 0:
        return 1
    return 0

print("⚙️ Đang tự động chấm điểm cho từng câu hỏi...")
df_merged['is_correct'] = df_merged.apply(cham_diem, axis=1)


# ==========================================
# 4. GOM NHÓM (NAMED AGGREGATION) VÀ GÁN NHÃN
# ==========================================
print("📦 Đang đóng gói dữ liệu thành các Chuỗi (Sequences)...")
df_sequences = df_merged.groupby(['session_id', 'user_id']).agg(
    seq_length=('question_id', 'count'),                         # Tính độ dài chuỗi
    total_time_response=('time_response (giây)', 'sum'),         # Tính tổng thời gian
    question_seq=('question_id', list),
    skill_seq=('skill_ids', list),                               # CHUỖI ĐỈNH KIẾN THỨC
    is_correct_seq=('is_correct', list),                         # CHUỖI ĐÚNG/SAI
    time_response_seq=('time_response (giây)', list),
    selected_options_seq=('options', list)
).reset_index()

# Gán nhãn trạng thái dựa trên thời gian (Dưới 5 phút = fast, Trên 4 tiếng = afk)
def xac_dinh_status(time_sec):
    if time_sec < 300: return 'abandoned_fast'
    elif time_sec > 14400: return 'abandoned_afk'
    else: return 'completed'

df_sequences['status'] = df_sequences['total_time_response'].apply(xac_dinh_status)

# Sắp xếp lại thứ tự các cột cho logic và chuẩn RDBMS
columns_order = [
    'session_id', 'user_id', 'status', 'seq_length', 'total_time_response',
    'question_seq', 'skill_seq', 'is_correct_seq', 'time_response_seq', 'selected_options_seq'
]
df_sequences = df_sequences[columns_order]


# ==========================================
# 5. XUẤT KẾT QUẢ ĐẦU RA CHO MÔ HÌNH
# ==========================================
output_file = 'AI_Training_Sequences.json'
df_sequences.to_json(output_file, orient='records', force_ascii=False, indent=4)

print("-" * 50)
print(f"✅ HOÀN TẤT XUẤT SẮC! Dữ liệu đã sẵn sàng cho Deep Learning.")
print(f"🔸 Tổng số Sessions gộp được: {len(df_sequences):,} phiên")
print(f"🔸 Số Session 'Sạch' (completed): {len(df_sequences[df_sequences['status'] == 'completed']):,} phiên")
print(f"💾 File lưu tại: {output_file}\n")

# In thử để bạn chiêm ngưỡng thành quả
print("🔍 XEM CẤU TRÚC 1 BẢN GHI HOÀN CHỈNH:")
sample_record = df_sequences.head(1).to_dict(orient='records')[0]
# Chỉ in 3 phần tử đầu của mỗi mảng cho đỡ dài màn hình
for key, val in sample_record.items():
    if isinstance(val, list) and len(val) > 3:
        print(f"  {key}: {val[:3]} ... (tổng {len(val)} phần tử)")
    else:
        print(f"  {key}: {val}")

🚀 BẮT ĐẦU XÂY DỰNG TẬP DỮ LIỆU HUẤN LUYỆN (SEQUENCES)...


C:\Users\Admin\AppData\Local\Temp\ipykernel_536\1283587212.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_history = pd.read_csv(r'D:\IntelligentTesting\intelligent-testing\dataset\Session.csv')


⚙️ Đang tự động chấm điểm cho từng câu hỏi...
📦 Đang đóng gói dữ liệu thành các Chuỗi (Sequences)...
--------------------------------------------------
✅ HOÀN TẤT XUẤT SẮC! Dữ liệu đã sẵn sàng cho Deep Learning.
🔸 Tổng số Sessions gộp được: 13,565 phiên
🔸 Số Session 'Sạch' (completed): 13,465 phiên
💾 File lưu tại: AI_Training_Sequences.json

🔍 XEM CẤU TRÚC 1 BẢN GHI HOÀN CHỈNH:
  session_id: 385
  user_id: 111170004
  status: abandoned_fast
  seq_length: 3
  total_time_response: 270
  question_seq: [7784, 7785, 7786]
  skill_seq: [[1], [8], [1]]
  is_correct_seq: [0, 1, 1]
  time_response_seq: [90, 90, 90]
  selected_options_seq: ['26266', '26267', '26272']


In [ ]:
import pandas as pd
import numpy as np

print("🚀 BẮT ĐẦU CHIA TẬP DỮ LIỆU BẰNG THUẬT TOÁN THAM LAM (GREEDY SPLIT)...")

# 1. TÍNH TOÁN TỈ LỆ ĐÚNG (ACCURACY)
df_sequences['accuracy'] = df_sequences['is_correct_seq'].apply(
    lambda x: sum(x) / len(x) if len(x) > 0 else 0
)

# 2. CHUẨN HÓA ĐẶC TRƯNG 
features = ['seq_length', 'total_time_response', 'accuracy']

# [FIX LỖI] ÉP KIỂU TOÀN BỘ VỀ FLOAT ĐỂ TRÁNH LỖI DTYPE('O')
X = df_sequences[features].astype(float).copy()
X_norm = (X - X.mean()) / X.std()

df_sequences['norm_len'] = X_norm['seq_length']
df_sequences['norm_time'] = X_norm['total_time_response']
df_sequences['norm_acc'] = X_norm['accuracy']

# 3. CẤU HÌNH TỈ LỆ CHIA (70% - 15% - 15%)
N = len(df_sequences)
targets = {
    'train': int(N * 0.70),
    'val': int(N * 0.15),
    'test': N - int(N * 0.70) - int(N * 0.15)
}

current_counts = {'train': 0, 'val': 0, 'test': 0}
current_sums = {'train': np.zeros(3, dtype=float), 'val': np.zeros(3, dtype=float), 'test': np.zeros(3, dtype=float)}

df_sequences['split'] = 'unassigned'

# 4. VÒNG LẶP THAM LAM
sorted_indices = df_sequences.sort_values('seq_length', ascending=False).index

print("⚖️ Đang phân bổ từng session vào các tập (Tối ưu hóa độ lệch trung bình)...")

for idx in sorted_indices:
    # [FIX LỖI] Đảm bảo mảng trích xuất ra là Float thuần chủng của Numpy
    session_feats = df_sequences.loc[idx, ['norm_len', 'norm_time', 'norm_acc']].values.astype(float)
    
    best_split = None
    min_cost = float('inf')
    
    for split in ['train', 'val', 'test']:
        if current_counts[split] >= targets[split]:
            continue
            
        proposed_count = current_counts[split] + 1
        proposed_mean = (current_sums[split] + session_feats) / proposed_count
        
        distance = np.linalg.norm(proposed_mean - np.zeros(3))
        
        if distance < min_cost:
            min_cost = distance
            best_split = split
            
    current_counts[best_split] += 1
    current_sums[best_split] += session_feats  # Lúc này += sẽ hoạt động mượt mà
    df_sequences.at[idx, 'split'] = best_split

print("✅ HOÀN TẤT CHIA TẬP!")

# Dọn dẹp cột thừa
df_sequences = df_sequences.drop(columns=['norm_len', 'norm_time', 'norm_acc'])

# ==========================================
# 5. XUẤT BÁO CÁO NGHIỆM THU
# ==========================================
report = df_sequences.groupby('split').agg(
    So_Luong_Session=('session_id', 'count'),
    Do_Dai_Trung_Binh=('seq_length', 'mean'),
    Thoi_Gian_Trung_Binh=('total_time_response', 'mean'),
    Ti_Le_Dung_Trung_Binh=('accuracy', lambda x: f"{x.mean()*100:.2f}%")
).reset_index()

tong_quat = pd.DataFrame([{
    'split': 'Global (Gốc)',
    'So_Luong_Session': len(df_sequences),
    'Do_Dai_Trung_Binh': df_sequences['seq_length'].mean(),
    'Thoi_Gian_Trung_Binh': df_sequences['total_time_response'].mean(),
    'Ti_Le_Dung_Trung_Binh': f"{df_sequences['accuracy'].mean()*100:.2f}%"
}])

bao_cao_cuoi = pd.concat([tong_quat, report], ignore_index=True)

print("\n📊 BẢNG NGHIỆM THU ĐỘ CÂN BẰNG CỦA 3 TẬP DỮ LIỆU:")
print(bao_cao_cuoi.to_string(index=False))

🚀 BẮT ĐẦU CHIA TẬP DỮ LIỆU BẰNG THUẬT TOÁN THAM LAM (GREEDY SPLIT)...
⚖️ Đang phân bổ từng session vào các tập (Tối ưu hóa độ lệch trung bình)...
✅ HOÀN TẤT CHIA TẬP!

📊 BẢNG NGHIỆM THU ĐỘ CÂN BẰNG CỦA 3 TẬP DỮ LIỆU:
       split  So_Luong_Session  Do_Dai_Trung_Binh  Thoi_Gian_Trung_Binh Ti_Le_Dung_Trung_Binh
Global (Gốc)             13565          46.775230           2795.804349                48.64%
        test              2036          53.006876           3054.059921                48.82%
       train              9495          44.701843           2707.681938                48.87%
         val              2034          50.216323           2948.662734                47.41%


In [ ]:
import pandas as pd

# 1. Vì bạn đã có df_sequences với cột 'split' chứa giá trị train, val, test
# Chúng ta chỉ cần đảm bảo dữ liệu đã được gán nhãn đầy đủ
# (Nếu bạn đang làm việc trên file đã lưu trước đó, hãy load lại nó)
# df_sequences = pd.read_json('AI_Training_Sequences_Fixed.json') 

# 2. Kiểm tra lại lần cuối để chắc chắn không còn giá trị 'unassigned'
# (Nếu có session nào chưa được chia, chúng ta sẽ gán tạm vào train)
df_sequences['split'] = df_sequences['split'].replace('unassigned', 'train')

# 3. Xuất toàn bộ ra file JSON duy nhất
output_final = 'AI_Training_Sequences_All_Split.json'
df_sequences.to_json(output_final, orient='records', force_ascii=False, indent=4)

print(f"✅ Đã đóng gói xong! Tổng số bản ghi: {len(df_sequences):,}")
print(f"💾 File được lưu tại: {output_final}")

# 4. In thống kê nhanh để bạn yên tâm
print("\n📊 Phân bổ dữ liệu trong file:")
print(df_sequences['split'].value_counts().to_string())

✅ Đã đóng gói xong! Tổng số bản ghi: 13,565
💾 File được lưu tại: AI_Training_Sequences_All_Split.json

📊 Phân bổ dữ liệu trong file:
split
train    9495
test     2036
val      2034
